# Previsão de Churn em Operadora de Telecomunicações

C318 – Fundamentos de Machine Learning (Inatel). Escopo e decisões: [`BACKLOG.md`](BACKLOG.md) e [`DECISION_LOG.md`](DECISION_LOG.md).

## 1. Dados e limpeza

- `customerID` é removido, pois é só um identificador.
- `TotalCharges` vem como texto; os 11 brancos são clientes com `tenure = 0` (ainda não pagaram nada) e recebem 0 (D-08).
- `Churn` vira 0/1 (1 = cancelou).
- **Não** padronizamos `SeniorCitizen` nem fundimos "No internet service" com "No": o one-hot encoding já trata essas colunas sem perda (D-09).

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_STATE = 42                      # mesma semente em todo o projeto
os.makedirs('figures', exist_ok=True)  # destino das figuras usadas nos slides

In [2]:
df = pd.read_csv('data/Telco-Customer-Churn.csv')
df = df.drop(columns='customerID')
df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
# TotalCharges vem como texto; os brancos (" ") viram NaN na conversão
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
nulos = df['TotalCharges'].isna()
n_nulos = nulos.sum()
tenure_dos_nulos = df.loc[nulos, 'tenure']
df['TotalCharges'] = df['TotalCharges'].fillna(0)  # D-08: cliente novo, ainda não pagou

df['Churn'] = (df['Churn'] == 'Yes').astype(int)   # alvo binário: 1 = cancelou

In [4]:
# Checagens de controle (D-07)
assert len(df) == 7043, 'número de clientes inesperado'
assert n_nulos == 11, 'número de brancos em TotalCharges inesperado'
assert (tenure_dos_nulos == 0).all(), 'brancos deveriam ser só clientes novos'
assert 0.26 < df['Churn'].mean() < 0.27, 'taxa de churn fora do esperado'
print(f"Clientes: {len(df)} | Taxa de churn: {df['Churn'].mean():.1%}")

Clientes: 7043 | Taxa de churn: 26.5%
